# Huấn luyện VEC trên Google Colab
Chọn **Runtime → Change runtime type → CPU** (GPU không giúp: mạng nhỏ, nút thắt là vòng lặp env).
Chi tiết từng bước: `HUONG_DAN_COLAB.md`.

## 1. Mount Drive (lưu kết quả qua các phiên)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/vec_paper1'   # thư mục trên Drive
!mkdir -p {DRIVE}

## 2. Lấy code (đã push lên GitHub) + dữ liệu Rio (upload `riobus_processed.tgz` vào thư mục Drive ở trên)

In [ ]:
%cd /content
!test -d LabEdgeComputing_Paper1 || git clone https://github.com/Linh126-minari/LabEdgeComputing_Paper1.git
%cd /content/LabEdgeComputing_Paper1
!git pull
!pip install -q -r requirements.txt
!tar xzf {DRIVE}/riobus_processed.tgz
!ls -la data/processed/riobus/
!nproc; free -g

## 3. Kiểm tra nhanh (~20 s)

In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -3

## 4. Chạy thử 1 seed để chọn số episode
Chạy ngầm để ô không bị treo; xem tiến độ ở ô kế tiếp. Chạy lại lệnh sau khi mất kết nối → tự tiếp tục các run chưa xong.

In [ ]:
RUNS = f'{DRIVE}/runs'
!python scripts/run_experiments.py --dry-run --seeds 0 --episodes 200 --maddpg-train-freq 4 --jobs 2 --speed-factor 1.8 --out-root {RUNS}
!nohup python scripts/run_experiments.py --seeds 0 --episodes 200 --maddpg-train-freq 4 --jobs 2 --out-root {RUNS} > {DRIVE}/launcher_seed0.log 2>&1 &

In [ ]:
!tail -n 20 {DRIVE}/launcher_seed0.log
!ls {RUNS}/*/*/seed0/agent_model.pth 2>/dev/null | wc -l   # 6 = xong

In [ ]:
!python scripts/plot_convergence.py --runs-dir {RUNS} --out {DRIVE}/convergence_seed0.png
from IPython.display import Image; Image(f'{DRIVE}/convergence_seed0.png')

## 5. Chạy đủ seed (seed 0 đã xong sẽ được bỏ qua — giữ nguyên `--episodes`)

In [ ]:
!nohup python scripts/run_experiments.py --seeds 0 1 2 --episodes 200 --maddpg-train-freq 4 --jobs 2 --out-root {RUNS} > {DRIVE}/launcher_all.log 2>&1 &

In [ ]:
!tail -n 20 {DRIVE}/launcher_all.log
!ls {RUNS}/*/*/seed*/agent_model.pth 2>/dev/null | wc -l   # 18 = xong

## 6. Đánh giá chung (synthetic & Rio test) từ model đã lưu — không train lại

In [ ]:
for a in ['lagrangian_dqn', 'dueling_dqn', 'maddpg']:
    !python scripts/sim2real_study.py --runs-dir {RUNS} --agent {a} --seeds 0 1 2 --eval-episodes 10 --output_dir {DRIVE}/eval 2>&1 | grep -A8 "Transfer matrix"
!ls {DRIVE}/eval